# Bag-of-Words
* CountVectorizer + MLP
* TF-IDF + MLP

In [1]:
import torch
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
metadata = pd.read_csv('../datasets/Flipkart/flipkart_com-ecommerce_sample_1050.csv')
count_vectorizer = CountVectorizer()
x_countvec = torch.tensor(count_vectorizer.fit_transform(metadata['description'].values).todense(), dtype=torch.float32)

tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
x_tfidfvec = torch.tensor(tfidf_vectorizer.fit_transform(metadata['description'].values).todense(), dtype=torch.float32)

print(x_countvec.shape)
print(x_tfidfvec.shape)

# Affichage compact du vocabulaire (on evite de dumper le dictionnaire complet,
# qui peut corrompre la sauvegarde du notebook a cause de la taille de l'output)
print(f"CountVectorizer : {len(count_vectorizer.vocabulary_)} tokens")
print(f"TF-IDF          : {len(tfidf_vectorizer.vocabulary_)} tokens")

torch.Size([1050, 6053])
torch.Size([1050, 500])
CountVectorizer : 6053 tokens
TF-IDF          : 500 tokens


In [3]:
le = LabelEncoder()
y = metadata['product_category_tree'].apply(lambda x: x.split(' >> ')[0][2:])
y = le.fit_transform(y)

y = torch.tensor(y, dtype=torch.long)

In [4]:
m = x_countvec.shape[0]

x_countvec /= x_countvec.amax(axis=1, keepdim=True)
x_tfidfvec /= x_tfidfvec.amax(axis=1, keepdim=True)

x_countvec = torch.hstack([x_countvec, torch.ones((m, 1))])
x_tfidfvec = torch.hstack([x_tfidfvec, torch.ones((m, 1))])

num_classes = y.unique().shape[0]
y = torch.nn.functional.one_hot(y, num_classes=num_classes).float()

In [5]:
class FlipkartDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader


class MLP(nn.Module):
    """Perceptron multicouche simple : Linear -> ReLU -> Linear."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, t):
        return self.network(t)


def train_mlp(X, y, hidden_dim=64, batch_size=32, n_epochs=200, eta=1e-1, verbose=True):
    """Entraine un MLP de maniere agnostique a la representation des features."""
    num_feats = X.shape[1]
    num_classes = y.shape[1]

    dataset = FlipkartDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = MLP(num_feats, hidden_dim, num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=eta)

    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in loader:
            outputs = model(X_batch)
            targets = torch.argmax(y_batch, dim=-1)
            loss = criterion(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            preds = torch.argmax(model(X), dim=-1)
            targets_all = torch.argmax(y, dim=-1)
            accuracy = (preds == targets_all).float().mean().item() * 100
            if verbose:
                print(f"Epoch {epoch + 1:3d}/{n_epochs} -- accuracy: {accuracy:.2f}%")

    return model


In [7]:
# CountVectorizer + MLP
model_countvec = train_mlp(x_countvec, y)

Epoch   1/200 -- accuracy: 51.24%
Epoch   2/200 -- accuracy: 58.95%
Epoch   3/200 -- accuracy: 64.48%
Epoch   4/200 -- accuracy: 69.90%
Epoch   5/200 -- accuracy: 80.95%
Epoch   6/200 -- accuracy: 84.57%
Epoch   7/200 -- accuracy: 88.76%
Epoch   8/200 -- accuracy: 90.76%
Epoch   9/200 -- accuracy: 92.57%
Epoch  10/200 -- accuracy: 94.00%
Epoch  11/200 -- accuracy: 94.19%
Epoch  12/200 -- accuracy: 94.95%
Epoch  13/200 -- accuracy: 95.71%
Epoch  14/200 -- accuracy: 96.10%
Epoch  15/200 -- accuracy: 96.29%
Epoch  16/200 -- accuracy: 96.00%
Epoch  17/200 -- accuracy: 96.86%
Epoch  18/200 -- accuracy: 96.86%
Epoch  19/200 -- accuracy: 96.95%
Epoch  20/200 -- accuracy: 96.95%
Epoch  21/200 -- accuracy: 97.33%
Epoch  22/200 -- accuracy: 97.71%
Epoch  23/200 -- accuracy: 97.71%
Epoch  24/200 -- accuracy: 97.81%
Epoch  25/200 -- accuracy: 98.10%
Epoch  26/200 -- accuracy: 98.00%
Epoch  27/200 -- accuracy: 98.00%
Epoch  28/200 -- accuracy: 98.00%
Epoch  29/200 -- accuracy: 98.10%
Epoch  30/200 

In [8]:
# TF-IDF + MLP
model_tfidf = train_mlp(x_tfidfvec, y)

Epoch   1/200 -- accuracy: 40.10%
Epoch   2/200 -- accuracy: 67.62%
Epoch   3/200 -- accuracy: 79.05%
Epoch   4/200 -- accuracy: 83.43%
Epoch   5/200 -- accuracy: 83.52%
Epoch   6/200 -- accuracy: 86.10%
Epoch   7/200 -- accuracy: 87.90%
Epoch   8/200 -- accuracy: 90.10%
Epoch   9/200 -- accuracy: 92.57%
Epoch  10/200 -- accuracy: 93.24%
Epoch  11/200 -- accuracy: 93.62%
Epoch  12/200 -- accuracy: 94.67%
Epoch  13/200 -- accuracy: 94.86%
Epoch  14/200 -- accuracy: 94.95%
Epoch  15/200 -- accuracy: 95.14%
Epoch  16/200 -- accuracy: 95.81%
Epoch  17/200 -- accuracy: 95.43%
Epoch  18/200 -- accuracy: 96.00%
Epoch  19/200 -- accuracy: 96.00%
Epoch  20/200 -- accuracy: 96.57%
Epoch  21/200 -- accuracy: 96.38%
Epoch  22/200 -- accuracy: 96.57%
Epoch  23/200 -- accuracy: 97.24%
Epoch  24/200 -- accuracy: 96.95%
Epoch  25/200 -- accuracy: 96.86%
Epoch  26/200 -- accuracy: 96.86%
Epoch  27/200 -- accuracy: 97.43%
Epoch  28/200 -- accuracy: 97.05%
Epoch  29/200 -- accuracy: 97.43%
Epoch  30/200 

# BERT + MLP sur Flipkart
Voir https://sbert.net/ 

Pour la liste des modèles préentrainés : https://sbert.net/docs/sentence_transformer/pretrained_models.html 

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(metadata['description'].values.tolist())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
embeddings.shape

(1050, 768)

In [11]:
# BERT embeddings + MLP (meme fonction, representation differente)
model_bert = train_mlp(torch.tensor(embeddings, dtype=torch.float), y)

Epoch   1/200 -- accuracy: 16.48%
Epoch   2/200 -- accuracy: 39.05%
Epoch   3/200 -- accuracy: 80.86%
Epoch   4/200 -- accuracy: 85.71%
Epoch   5/200 -- accuracy: 83.05%
Epoch   6/200 -- accuracy: 83.81%
Epoch   7/200 -- accuracy: 83.05%
Epoch   8/200 -- accuracy: 86.19%
Epoch   9/200 -- accuracy: 86.76%
Epoch  10/200 -- accuracy: 89.43%
Epoch  11/200 -- accuracy: 89.71%
Epoch  12/200 -- accuracy: 90.57%
Epoch  13/200 -- accuracy: 91.14%
Epoch  14/200 -- accuracy: 91.05%
Epoch  15/200 -- accuracy: 91.52%
Epoch  16/200 -- accuracy: 91.52%
Epoch  17/200 -- accuracy: 92.10%
Epoch  18/200 -- accuracy: 92.38%
Epoch  19/200 -- accuracy: 92.48%
Epoch  20/200 -- accuracy: 92.48%
Epoch  21/200 -- accuracy: 92.95%
Epoch  22/200 -- accuracy: 92.76%
Epoch  23/200 -- accuracy: 93.14%
Epoch  24/200 -- accuracy: 93.62%
Epoch  25/200 -- accuracy: 93.62%
Epoch  26/200 -- accuracy: 94.00%
Epoch  27/200 -- accuracy: 93.90%
Epoch  28/200 -- accuracy: 94.19%
Epoch  29/200 -- accuracy: 94.57%
Epoch  30/200 

# Full Transformers sur Flipkart

In [12]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

# ===========================
# Parameters
# ===========================
MAX_SEQ_LEN = 16
EMBED_DIM = 128
NUM_HEADS = 2
FF_DIM = 32
NUM_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 8
NUM_CLASSES = 7
LR = 1e-3
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

texts = metadata['description'].values.tolist()

y_tensor = torch.argmax(y, dim=1)

# ===========================
# Tokenizer + vocab
# ===========================
def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text.split()

all_tokens = [token for text in texts for token in simple_tokenizer(text)]
vocab = {token: i+1 for i, (token, _) in enumerate(Counter(all_tokens).most_common())}
vocab_size = len(vocab) + 1

def encode(text):
    tokens = simple_tokenizer(text)
    ids = [vocab.get(tok,0) for tok in tokens]
    if len(ids) < MAX_SEQ_LEN:
        ids += [0]*(MAX_SEQ_LEN - len(ids))
    else:
        ids = ids[:MAX_SEQ_LEN]
    return ids

X_tensor = torch.tensor([encode(t) for t in texts], dtype=torch.long)

# ===========================
# Dataset & DataLoader
# ===========================
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = TextDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ===========================
# Positional Encoding
# ===========================
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=MAX_SEQ_LEN):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / embed_dim))
        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(1))  # (max_len, 1, embed_dim)

    def forward(self, x):
        # x: (seq_len, batch, embed_dim)
        x = x + self.pe[:x.size(0), :]
        return x

# ===========================
# Transformer Encoder Model using nn.TransformerEncoder
# ===========================
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, num_classes, max_len=MAX_SEQ_LEN, dropout=DROPOUT):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pos_embed = PositionalEncoding(embed_dim, max_len)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        x = self.embed(x)                  # (batch, seq_len, embed_dim)
        x = x.transpose(0,1)               # (seq_len, batch, embed_dim)
        x = self.pos_embed(x)              # add positional encoding
        x = self.transformer_encoder(x)    # (seq_len, batch, embed_dim)
        x = x.mean(dim=0)                  # global average pooling over sequence
        out = self.classifier(x)
        return out

# ===========================
# Instantiate model
# ===========================
model = TransformerClassifier(vocab_size=vocab_size,
                              embed_dim=EMBED_DIM,
                              num_heads=NUM_HEADS,
                              ff_dim=FF_DIM,
                              num_layers=NUM_LAYERS,
                              num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

from sklearn.model_selection import train_test_split

# ===========================
# Split train/validation
# ===========================
X_train, X_val, y_train, y_val = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42, stratify=y_tensor
)

train_dataset = TextDataset(X_train, y_train)
val_dataset = TextDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ===========================
# Training loop with validation accuracy
# ===========================
for epoch in range(EPOCHS):
    # --- training ---
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
    
    train_loss = total_loss / len(train_loader)
    train_acc = correct / total * 100

    # --- validation ---
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss_total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss_total += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += y_batch.size(0)
    
    val_loss = val_loss_total / len(val_loader)
    val_acc = val_correct / val_total * 100

    print(f"Epoch {epoch+1}/{EPOCHS}, "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")


C:\Users\chaki\AppData\Local\Temp\ipykernel_23128\423159541.py:90: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Epoch 1/30, Train Loss: 1.0609, Train Acc: 62.26%, Val Loss: 0.7192, Val Acc: 71.90%
Epoch 2/30, Train Loss: 0.4213, Train Acc: 86.90%, Val Loss: 0.5342, Val Acc: 81.90%
Epoch 3/30, Train Loss: 0.2276, Train Acc: 92.26%, Val Loss: 0.5546, Val Acc: 82.86%
Epoch 4/30, Train Loss: 0.1363, Train Acc: 95.36%, Val Loss: 0.4374, Val Acc: 88.57%
Epoch 5/30, Train Loss: 0.0801, Train Acc: 97.98%, Val Loss: 0.5469, Val Acc: 86.67%
Epoch 6/30, Train Loss: 0.0596, Train Acc: 98.33%, Val Loss: 0.5452, Val Acc: 88.57%
Epoch 7/30, Train Loss: 0.0435, Train Acc: 98.21%, Val Loss: 0.5148, Val Acc: 87.14%
Epoch 8/30, Train Loss: 0.0422, Train Acc: 98.93%, Val Loss: 0.6899, Val Acc: 87.14%
Epoch 9/30, Train Loss: 0.0938, Train Acc: 96.67%, Val Loss: 0.5960, Val Acc: 85.24%
Epoch 10/30, Train Loss: 0.0849, Train Acc: 97.02%, Val Loss: 0.5842, Val Acc: 86.67%
Epoch 11/30, Train Loss: 0.1394, Train Acc: 95.95%, Val Loss: 0.5655, Val Acc: 85.71%
Epoch 12/30, Train Loss: 0.0650, Train Acc: 97.98%, Val Loss: 0

# LSTM sur Flipkart

Avant les Transformers, les **reseaux recurrents** (RNN, LSTM, GRU) etaient l'outil de
reference pour les sequences. Un LSTM lit les tokens un a un en maintenant un **etat
cache** ; on utilise le dernier etat cache comme resume de la sequence pour classifier.

On reprend ici **exactement la meme tache et la meme tokenisation** que pour le Transformer
(categorisation produit Flipkart) afin de **comparer** une approche recurrente a
l'attention.

In [13]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

# On reutilise X_tensor, y_tensor, vocab_size, TextDataset, EMBED_DIM, NUM_CLASSES,
# BATCH_SIZE, EPOCHS, LR, DEVICE... definis dans la cellule Transformer (meme
# tokenisation). On reconstruit simplement les dataloaders.
X_train, X_val, y_train, y_val = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42, stratify=y_tensor
)
train_loader_lstm = DataLoader(TextDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader_lstm   = DataLoader(TextDataset(X_val, y_val), batch_size=BATCH_SIZE)


class LSTMClassifier(nn.Module):
    """Embedding -> LSTM -> dernier etat cache -> couche lineaire."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embed(x)                # (batch, seq_len, embed_dim)
        out, (h, c) = self.lstm(emb)       # h: (num_layers, batch, hidden_dim)
        resume = h[-1]                     # dernier etat cache : (batch, hidden_dim)
        return self.classifier(resume)


lstm_model = LSTMClassifier(vocab_size, EMBED_DIM, hidden_dim=128, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    lstm_model.train()
    total_loss = 0
    correct = 0
    total = 0
    for X_batch, y_batch in train_loader_lstm:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        outputs = lstm_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == y_batch).sum().item()
        total += y_batch.size(0)
    train_loss = total_loss / len(train_loader_lstm)
    train_acc = correct / total * 100

    lstm_model.eval()
    val_correct = 0
    val_total = 0
    val_loss_total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader_lstm:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            outputs = lstm_model(X_batch)
            val_loss_total += criterion(outputs, y_batch).item()
            val_correct += (outputs.argmax(1) == y_batch).sum().item()
            val_total += y_batch.size(0)
    val_loss = val_loss_total / len(val_loader_lstm)
    val_acc = val_correct / val_total * 100

    print(f"Epoch {epoch+1}/{EPOCHS}, "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")


Epoch 1/30, Train Loss: 1.4322, Train Acc: 50.71%, Val Loss: 0.9710, Val Acc: 66.67%
Epoch 2/30, Train Loss: 0.5396, Train Acc: 82.74%, Val Loss: 0.5309, Val Acc: 84.29%
Epoch 3/30, Train Loss: 0.2113, Train Acc: 94.17%, Val Loss: 0.5771, Val Acc: 81.90%
Epoch 4/30, Train Loss: 0.1402, Train Acc: 95.12%, Val Loss: 0.4473, Val Acc: 85.24%
Epoch 5/30, Train Loss: 0.0891, Train Acc: 97.02%, Val Loss: 0.4958, Val Acc: 84.76%
Epoch 6/30, Train Loss: 0.0389, Train Acc: 98.93%, Val Loss: 0.5259, Val Acc: 86.19%
Epoch 7/30, Train Loss: 0.0250, Train Acc: 99.52%, Val Loss: 0.5866, Val Acc: 85.24%
Epoch 8/30, Train Loss: 0.0174, Train Acc: 99.52%, Val Loss: 0.6697, Val Acc: 84.29%
Epoch 9/30, Train Loss: 0.0930, Train Acc: 97.62%, Val Loss: 0.5505, Val Acc: 85.71%
Epoch 10/30, Train Loss: 0.0303, Train Acc: 99.40%, Val Loss: 0.5970, Val Acc: 87.14%
Epoch 11/30, Train Loss: 0.0177, Train Acc: 99.29%, Val Loss: 0.7124, Val Acc: 84.29%
Epoch 12/30, Train Loss: 0.0080, Train Acc: 99.76%, Val Loss: 0

# CNN vs ViT

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

# ===========================
# Hyperparameters
# ===========================
batch_size = 128
num_classes = 10
num_epochs = 3
learning_rate = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===========================
# Dataset
# ===========================
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# ===========================
# LeNet5_GAP definition
# ===========================
class LeNet5_GAP(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5_GAP, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5)  # 3 channels for CIFAR10
        self.pool1 = nn.AvgPool2d(2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.AvgPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        x = torch.tanh(self.conv1(x))
        x = self.pool1(x)
        x = torch.tanh(self.conv2(x))
        x = self.pool2(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# ===========================
# ViT definition (8x8 patches)
# ===========================
import torch
import torch.nn as nn
import torch.nn.functional as F

# ===========================
# Transformer Encoder Block
# ===========================
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (seq_len, batch, embed_dim)
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.dropout(attn_out)
        x_norm = self.norm2(x)
        x = x + self.mlp(x_norm)
        return x

# ===========================
# ViT Model
# ===========================
class ViT(nn.Module):
    def __init__(self, image_size=32, patch_size=8, in_channels=3, num_classes=10,
                 embed_dim=128, depth=6, num_heads=4, mlp_dim=256, dropout=0.1):
        super().__init__()
        assert image_size % patch_size == 0, "Image size must be divisible by patch size"
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_size = patch_size
        self.embed_dim = embed_dim

        # Patch embedding
        self.patch_embed = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        # Transformer encoder
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Classifier head
        self.head = nn.Linear(embed_dim, num_classes)

        # Initialize weights
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        # x: (B, C, H, W)
        B = x.shape[0]
        x = self.patch_embed(x)  # (B, embed_dim, H/patch, W/patch)
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)

        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat((cls_tokens, x), dim=1)          # (B, num_patches+1, embed_dim)
        x = x + self.pos_embed
        x = self.dropout(x)

        # Transformer blocks (need seq_len, batch, embed_dim for nn.MultiheadAttention)
        x = x.transpose(0, 1)  # (seq_len, batch, embed_dim)
        for block in self.blocks:
            x = block(x)
        x = x.transpose(0, 1)  # (B, seq_len, embed_dim)

        x = self.norm(x[:, 0])  # Take CLS token
        x = self.head(x)
        return x

vit_model = ViT(image_size=32, patch_size=8, in_channels=3, num_classes=10)


# ===========================
# Training and evaluation functions
# ===========================
def train_model(model, criterion, optimizer, num_epochs):
    model = model.to(device)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f} - Time: {time.time()-start_time:.1f}s")

def evaluate_model(model):
    model.eval()
    model = model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    acc = correct / total
    print(f"Test Accuracy: {acc:.4f}")
    return acc

# ===========================
# Train and evaluate LeNet5_GAP
# ===========================
print("=== Training LeNet5_GAP ===")
lenet_model = LeNet5_GAP(num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lenet_model.parameters(), lr=learning_rate)

train_model(lenet_model, criterion, optimizer, num_epochs)
acc_lenet = evaluate_model(lenet_model)

# ===========================
# Train and evaluate ViT
# ===========================
print("\n=== Training ViT (8x8 patches) ===")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vit_model.parameters(), lr=learning_rate)

train_model(vit_model, criterion, optimizer, num_epochs)
acc_vit = evaluate_model(vit_model)

print(f"\nLeNet5_GAP Accuracy: {acc_lenet:.4f}")
print(f"ViT Accuracy: {acc_vit:.4f}")


100%|██████████| 170M/170M [00:26<00:00, 6.32MB/s] 


=== Training LeNet5_GAP ===
Epoch [1/3] - Loss: 2.0573 - Acc: 0.2258 - Time: 16.0s
Epoch [2/3] - Loss: 1.9397 - Acc: 0.2835 - Time: 15.1s
Epoch [3/3] - Loss: 1.8945 - Acc: 0.3050 - Time: 16.1s
Test Accuracy: 0.3406

=== Training ViT (8x8 patches) ===
Epoch [1/3] - Loss: 1.8840 - Acc: 0.2892 - Time: 20.1s
Epoch [2/3] - Loss: 1.6979 - Acc: 0.3637 - Time: 20.1s
Epoch [3/3] - Loss: 1.6047 - Acc: 0.4077 - Time: 19.9s
Test Accuracy: 0.4500

LeNet5_GAP Accuracy: 0.3406
ViT Accuracy: 0.4500
